# 🚀 Day 8 — RAG Evaluation

### Goal

Today we're answering:

> **How do we know whether our RAG system is actually good?**

We'll build a small RAG system and evaluate it at multiple levels:

```text
                    RAG SYSTEM
                        │
          ┌─────────────┴─────────────┐
          ↓                           ↓
      RETRIEVAL                   GENERATION
          │                           │
    Recall@K                    Correctness
    Precision@K                 Relevance
    Context relevance           Faithfulness
          │                           │
          └─────────────┬─────────────┘
                        ↓
                 END-TO-END QUALITY
```

We'll also learn **LLM-as-a-Judge** and its limitations.

---

# 🟢 CELL 1 — Install dependencies

In [42]:
!pip -q install chromadb sentence-transformers langchain-google-genai

---

# 🟢 CELL 2 — Imports

In [ ]:
import chromadb

from sentence_transformers import SentenceTransformer

from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI

---

# 🟢 CELL 3 — API key

In [ ]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("API key loaded:", GEMINI_API_KEY is not None)

API key loaded: True


Expected:

```text
API key loaded: True
```

---

# 🟢 CELL 4 — Load models

We're using the same lightweight embedding model you've already worked with.

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY
)

---

# 🟢 CELL 5 — Create our knowledge base

We'll intentionally keep this small so that we can understand the evaluation mechanics.

In [ ]:
documents = [
    {
        "id": "generator_memory",
        "text": """
        Python generators produce values lazily. Instead of creating all values
        in memory at once, a generator produces values one at a time when they
        are requested. This makes generators useful for processing large
        datasets while reducing memory usage.
        """
    },

    {
        "id": "generator_yield",
        "text": """
        Python generator functions use the yield keyword. Calling a generator
        function returns a generator object. Execution pauses at each yield
        and resumes when another value is requested.
        """
    },

    {
        "id": "fastapi",
        "text": """
        FastAPI is a Python web framework designed for building APIs. It uses
        Python type hints for request validation and can automatically generate
        OpenAPI documentation.
        """
    },

    {
        "id": "redis",
        "text": """
        Redis is an in-memory data store commonly used for caching, queues,
        sessions, and other applications that require fast data access.
        """
    },

    {
        "id": "docker",
        "text": """
        Docker packages an application together with its dependencies into
        containers. Containers help applications run consistently across
        different environments.
        """
    }
]

print("Number of documents:", len(documents))

Number of documents: 5


Expected:

```text
Number of documents: 5
```

---

# 🟢 CELL 6 — Prepare texts and IDs

In [ ]:
texts = [doc["text"] for doc in documents]
ids = [doc["id"] for doc in documents]

print("IDs:")
print(ids)

IDs:
['generator_memory', 'generator_yield', 'fastapi', 'redis', 'docker']


You should see:

```text
[
 'generator_memory',
 'generator_yield',
 'fastapi',
 'redis',
 'docker'
]
```

---

# 🟢 CELL 7 — Create embeddings

In [ ]:
document_embeddings = embedding_model.encode(texts)

print("Number of embeddings:", len(document_embeddings))
print("Embedding dimensions:", len(document_embeddings[0]))

Number of embeddings: 5
Embedding dimensions: 384


Expected approximately:

```text
Number of embeddings: 5
Embedding dimensions: 384
```

---

# 🟢 CELL 8 — Create Chroma collection

In [ ]:
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="day8_evaluation"
)

---

# 🟢 CELL 9 — Store documents

In [ ]:
collection.add(
    ids=ids,
    documents=texts,
    embeddings=document_embeddings.tolist()
)

print("Documents stored:", collection.count())

Documents stored: 5


Expected:

```text
Documents stored: 5
```

---

# 🟢 CELL 10 — Test retrieval

Before evaluating anything, make sure retrieval works.

In [ ]:
question = "How do Python generators save memory?"

query_embedding = embedding_model.encode(
    [question]
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "distances"]
)

Print:

In [ ]:
for i, (doc, distance) in enumerate(
    zip(
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"{i}. Distance: {distance:.4f}")
    print(doc)
    print()


1. Distance: 0.4021

        Python generators produce values lazily. Instead of creating all values
        in memory at once, a generator produces values one at a time when they
        are requested. This makes generators useful for processing large
        datasets while reducing memory usage.
        

2. Distance: 0.6519

        Python generator functions use the yield keyword. Calling a generator
        function returns a generator object. Execution pauses at each yield
        and resumes when another value is requested.
        

3. Distance: 1.4626

        Redis is an in-memory data store commonly used for caching, queues,
        sessions, and other applications that require fast data access.
        



Remember:

> **Smaller distance = closer according to the configured metric.**

---

# 🟢 CELL 11 — Build the RAG function

Now we're creating the actual RAG pipeline.

In [ ]:
def rag_answer(question, top_k=3):

    # Step 1: Convert question into an embedding
    query_embedding = embedding_model.encode(
        [question]
    )[0]

    # Step 2: Retrieve documents
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        include=["documents", "distances"]
    )

    retrieved_documents = results["documents"][0]
    retrieved_distances = results["distances"][0]

    # Step 3: Build context
    context = "\n\n".join(retrieved_documents)

    # Step 4: Grounded prompt
    prompt = f"""
You are a helpful AI assistant.

Answer the question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Do not use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 5: Generate answer
    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "documents": retrieved_documents,
        "distances": retrieved_distances
    }

---

# 🟢 CELL 12 — Test the RAG system

In [43]:
result = rag_answer(
    "How do Python generators save memory?"
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")

for doc, distance in zip(
    result["documents"],
    result["distances"]
):
    print("\nDistance:", round(distance, 4))
    print(doc)

ANSWER:
[{'type': 'text', 'text': 'Python generators save memory by producing values lazily—one at a time when they are requested—instead of creating all values in memory at once.', 'extras': {'signature': 'EsILCr8LARFNMg/1HKFeJeEVf2s5VASn3bUajJdu4MwDhgF2+q6+cemuh8puEMyZT9qnjIQd4NkW3CqwxR5END1EOeVvlg+y6heE1Sj3yMJ51lXIH8v96RaZWadtkvi+lZYI8Ksc0yIwl1v0J/WEWhssmNVj/nKKxfB9fTGDMMOSFzxxY04T7LJqlrPmQw2+mibXdJACX0Uv3nWG3aBndswCWyRjLaMtcj+oQuT976b4eA/ACtBCLxtcCJ6iaqYKSYRUApbsj9AQ1JXDlXYiaW6fh8DVWiBzNZj4goUwpwJI9zuSWlJcx0EjXD2r7f0jFf4XnXHFz1B24fWePhqID+WOMmmaOlSKL3fOGlZGHl/KF4wQskC0H5gpqwX1iepGme51dzu5hsDITD1rf10OwBE73rzJ/ZSenK7Mts8A61cl9UEC3M6skkhwbfqIyLIjMi9P8mkdc8C6DW/RX5ob+0FpT7ofksPxxWwi2TmENaLssxpvEhgR2obeVXgj3E/cgU+dVcejAZ1B0XDBAz0FO3Gc1CgaVjTxi/xH/vvDoAC4Y2Bj+Y0qq492YoIqhD/Uw+if2mzJqJIPlI33aI0qCpNfZTYE7yYEyi3ccxU2U64kbrbMO35/dYEn6M3hWE3Rm8Qnjfu0UV8ICvrgahDptTRBOCHlBKb+PIHDTBPu9i3VmhyQvOpBVVXJ3FQeYuYWSSzSfOkD8oWvItytBGxqY7ap8I/vZm1I1UJ1H7F4U79dKg052VBmRLSPwrvz5RCrgB7N9BL7by6t1wGlUTVo9uKrc

If this works, our RAG system is ready.

---

# 🟡 PART 2 — BUILD THE EVALUATION DATASET

Now comes the actual Day 8 concept.

## What is an evaluation dataset?

Instead of randomly asking questions, we create known test cases.

Each test case contains:

```text
Question
Relevant source
Expected answer
```

This is sometimes called a **golden dataset**.

---

# 🟢 CELL 13 — Evaluation questions

In [44]:
eval_questions = [
    {
        "question": "How do Python generators save memory?",
        "relevant_sources": ["generator_memory"],
        "expected_answer":
            "Generators save memory by producing values lazily "
            "instead of creating all values in memory at once."
    },

    {
        "question": "What keyword is used to create generator functions?",
        "relevant_sources": ["generator_yield"],
        "expected_answer":
            "The yield keyword is used to create generator functions."
    },

    {
        "question": "What is FastAPI used for?",
        "relevant_sources": ["fastapi"],
        "expected_answer":
            "FastAPI is a Python web framework used for building APIs."
    },

    {
        "question": "Why is Redis useful?",
        "relevant_sources": ["redis"],
        "expected_answer":
            "Redis is useful for caching, queues, sessions, and other "
            "applications requiring fast data access."
    },

    {
        "question": "What does Docker package?",
        "relevant_sources": ["docker"],
        "expected_answer":
            "Docker packages an application and its dependencies into containers."
    }
]

print("Evaluation questions:", len(eval_questions))

Evaluation questions: 5


---

# 🔵 PART 3 — RETRIEVAL EVALUATION

Now we're going to evaluate the **retriever independently**.

This is important.

We don't want to blame the LLM for a retrieval problem.

---

# 🟢 CELL 14 — Recall@K

Recall@K asks:

> **Did we retrieve the relevant document within the top K results?**

Formula:

```text
Recall@K =
questions where relevant information appeared in top K
-------------------------------------------------------
total evaluation questions
```

Code:

In [45]:
def evaluate_recall_at_k(eval_questions, k=3):

    correct = 0

    for item in eval_questions:

        question = item["question"]
        relevant_sources = set(item["relevant_sources"])

        query_embedding = embedding_model.encode(
            [question]
        )[0]

        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )

        retrieved_ids = results["ids"][0]

        if relevant_sources.intersection(retrieved_ids):
            correct += 1

    return correct / len(eval_questions)

---

# 🟢 CELL 15 — Test Recall

In [46]:
for k in [1, 2, 3, 5]:

    recall = evaluate_recall_at_k(
        eval_questions,
        k=k
    )

    print(f"Recall@{k}: {recall:.2f}")

Recall@1: 1.00
Recall@2: 1.00
Recall@3: 1.00
Recall@5: 1.00


You may see something like:

```text
Recall@1: 0.80
Recall@2: 1.00
Recall@3: 1.00
Recall@5: 1.00
```

Your numbers may differ.

### Interpretation

If:

```text
Recall@3 = 1.0
```

that means:

> For every evaluation question, at least one relevant document appeared in the top 3.

It **doesn't** mean the final answers are perfect.

---

# 🟢 CELL 16 — Precision@K

Precision@K asks:

> **Of the documents we retrieved, how many were actually relevant?**

Formula:

```text
Precision@K =
relevant retrieved documents
----------------------------
total retrieved documents
```

Code:

In [47]:
def evaluate_precision_at_k(eval_questions, k=3):

    total_precision = 0

    for item in eval_questions:

        question = item["question"]
        relevant_sources = set(item["relevant_sources"])

        query_embedding = embedding_model.encode(
            [question]
        )[0]

        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )

        retrieved_ids = results["ids"][0]

        relevant_retrieved = 0

        for doc_id in retrieved_ids:

            if doc_id in relevant_sources:
                relevant_retrieved += 1

        precision = relevant_retrieved / k

        total_precision += precision

    return total_precision / len(eval_questions)

---

# 🟢 CELL 17 — Test Precision

In [48]:
for k in [1, 2, 3, 5]:

    precision = evaluate_precision_at_k(
        eval_questions,
        k=k
    )

    print(f"Precision@{k}: {precision:.2f}")

Precision@1: 1.00
Precision@2: 0.50
Precision@3: 0.33
Precision@5: 0.20


---

# 🧠 Recall vs Precision

Remember this forever:

```text
RECALL

"Did I find the relevant information?"
```

versus:

```text
PRECISION

"Of what I retrieved, how much was relevant?"
```

Example:

```text
Top 5 retrieved:

A ✅
B ❌
C ❌
D ❌
E ❌
```

If A is relevant:

```text
Precision@5 = 1/5 = 20%
```

But if A is the only relevant document:

```text
Recall@5 = 1/1 = 100%
```

So you can have:

```text
High Recall
+
Low Precision
```

Meaning:

> We found the answer, but we're also retrieving a lot of irrelevant stuff.

That can hurt downstream generation.

---

# 🟣 PART 4 — GENERATE ANSWERS FOR ALL TEST CASES

Now we evaluate the **LLM side**.

---

# 🟢 CELL 18 — Run all evaluation questions

In [49]:
eval_results = []

for item in eval_questions:

    result = rag_answer(
        item["question"],
        top_k=3
    )

    eval_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "generated_answer": result["answer"],
        "sources": result["documents"]
    })

print("Generated answers:", len(eval_results))

Generated answers: 5


---

# 🟢 CELL 19 — Inspect the answers

In [50]:
for i, result in enumerate(
    eval_results,
    start=1
):

    print("=" * 70)
    print(f"QUESTION {i}")
    print("=" * 70)

    print("\nQUESTION:")
    print(result["question"])

    print("\nEXPECTED ANSWER:")
    print(result["expected_answer"])

    print("\nGENERATED ANSWER:")
    print(result["generated_answer"])

    print("\nRETRIEVED CONTEXT:")
    for source in result["sources"]:
        print(source)

    print()

QUESTION 1

QUESTION:
How do Python generators save memory?

EXPECTED ANSWER:
Generators save memory by producing values lazily instead of creating all values in memory at once.

GENERATED ANSWER:
[{'type': 'text', 'text': 'Python generators reduce memory usage by producing values lazily—one at a time when they are requested—instead of creating all values in memory at once.', 'extras': {'signature': 'EqoNCqcNARFNMg8tPN5P59GuRyx59HU09g85YthYKnH4EtEyiwV7D9e8ojxxuiDJyY6f26yvYk0PaxxllRNytzbONBP0whpEv/Ue2tnzkOlWPfLntJHNXdDBB+UftPxoGut+yTVqdJUvAuk79E4V6524ls/WJixX0OUm44DGEXRouE0k7f2yJh+MeEnHp4SZgvW9ko5gxSDje9D8tlNkCvOdmswbOaZD/2vo2OBy9ij5e7A/m5VJ/3UPvItya43OOt9Mp5PfR6aK1NKqIPvKpqa3AruNZstrKDDBeotc4F78Z3tw+YAzWS7sNcMo6EQnpfoHw6TO098b6peKBqYucaPlpt6JkLs9yn8X4Fa4IiuUVqnFppdBTcssHx2BUDXuot+HqL0ua+gWuc995nvjT9UOralBK4AO4RVVzHLMhbj87br3Up/epeRn1UdhfzWUiwyyWCvG6/wViWT8JvMYUiXwFHvXhPHQD7pLZK/osINkeG5E6e33adFlM1Z06bQ0b3CUB3VFPTPejIjheQToh34ueBJm0lXwALpgMtqtM/nTfBlNZ4TI4JWn54IxKXNXSwGsDk9NwwdL3A5hUNXa

Now you can actually inspect what your RAG system is doing.

---

# 🔴 PART 5 — THREE ANSWER METRICS

This is where Day 8 becomes important.

## 1. Correctness

> Is the answer factually correct?

Example:

```text
Expected:
Generators produce values lazily.

Generated:
Generators produce values lazily.
```

✅ Correct.

---

## 2. Relevance

> Does the answer actually answer the question?

Question:

> What is Redis used for?

Answer:

> Redis was created in 2009.

Even if that fact were true:

```text
Correctness → potentially ✅
Relevance → ❌
```

It doesn't answer the question.

---

## 3. Faithfulness

> Is the answer supported by the retrieved context?

Context:

```text
Generators produce values lazily.
```

Answer:

```text
Generators produce values lazily and are
always 10x faster than lists.
```

The "10x faster" claim isn't in the context.

Therefore:

```text
Faithfulness → ❌
```

---

# 🟢 CELL 20 — Manual evaluation

For each answer, give:

```text
Correctness:
1 = incorrect
2 = partially correct
3 = correct

Relevance:
1 = doesn't answer
2 = partially answers
3 = directly answers

Faithfulness:
1 = unsupported
2 = partially supported
3 = fully supported
```

Create:

In [51]:
manual_scores = [
    {
        "question": eval_results[0]["question"],
        "correctness": 3,
        "relevance": 3,
        "faithfulness": 3
    },

    {
        "question": eval_results[1]["question"],
        "correctness": 3,
        "relevance": 3,
        "faithfulness": 3
    },

    {
        "question": eval_results[2]["question"],
        "correctness": 3,
        "relevance": 3,
        "faithfulness": 3
    },

    {
        "question": eval_results[3]["question"],
        "correctness": 3,
        "relevance": 3,
        "faithfulness": 3
    },

    {
        "question": eval_results[4]["question"],
        "correctness": 3,
        "relevance": 3,
        "faithfulness": 3
    }
]

**Don't blindly put 3.**

Actually inspect your generated answers and change the numbers based on what you see.

---

# 🟢 CELL 21 — Calculate average manual scores

In [52]:
avg_correctness = sum(
    x["correctness"]
    for x in manual_scores
) / len(manual_scores)

avg_relevance = sum(
    x["relevance"]
    for x in manual_scores
) / len(manual_scores)

avg_faithfulness = sum(
    x["faithfulness"]
    for x in manual_scores
) / len(manual_scores)

print("Average Correctness:", round(avg_correctness, 2))
print("Average Relevance:", round(avg_relevance, 2))
print("Average Faithfulness:", round(avg_faithfulness, 2))

Average Correctness: 3.0
Average Relevance: 3.0
Average Faithfulness: 3.0


---

# 🟠 PART 6 — INSUFFICIENT INFORMATION

This is extremely important for RAG.

A good RAG system should know when it **doesn't have the answer**.

Let's test:

In [53]:
question = "Who invented Docker?"

result = rag_answer(
    question,
    top_k=3
)

print(result["answer"])

[{'type': 'text', 'text': "I don't have enough information in the provided documents.", 'extras': {'signature': 'EsMKCsAKARFNMg/RqDK7Fx8HzZdbnmhU+HiPl+PlKejvBvW4tnznRQAdK+NHZSzjQnmrhUA7N6EjYMcDXH/xzfNLXjkhCydWPXrvWt6bvgqbTvIjhwItunmq2JOe/yuE/s0GyNlETtZ7ztXOXlKZyNjToyw3uiGhIPhpBUWNRSrhRC1n9Y7mfyU2VZ+WPwbkfVcjwQbyFDB0yRBIeiKBKJdQthBebMkqDxRyo3kVl2aryEAYoHW2vQNXJH0j57B8H1sc3BMvi/HDLkxRok4c0LjAKjkfXE6ZiChMQ7k6zZ2S8x/aiPv4dnylH+AOuE8o/RBMCoG+qsW//T6o341ZnRnL+tJL5UR/1QOLaqPZFc/l3Evgo3V4RTc2ZC7bZvtW1GOixlhWE5h+PqqDEUKc1Z8EBg+6tcIfG+PhdG6e2Pc8sfWHVtioURrCvr/ZPWVdgqEKaJB9EQlMG9BHdsj75f1gC120jRW+OWGvCGEAxraW9p7i7I67UGKmgTOyqIMD3qXMsD6BZx1iaHcZAz8Ui0F2zyrqhRWeiFcrWsHIT77RhCtObNDmgGgYxJRHbiVMxIJInIl8eDfANEyOfA1ZK86buLuusgcQcEsfC6zfHByhN1eIcTitH5NoFu5akGN7MTYVjXNpI6TEKnUimDzIOqYmwk8dAWJaU7vNEgRqnpA7idn41+wO84VhoveTMxQCjt7FvNzvi249sbmged/9vPnVv5UJvEh7uM0cEPSJw54hP5/Q+Hu/kwKSO1+p4hUMcuMqOXqPB3SWolHKfwZA4HWGAywIQ+0Xekv85kqa2V/HP55Lx895K3foBuutYN3F4fj6V3MquI5pYmHSmhLFQXQe/qK3iskgjqsPSIblLoKijDmNrQLfx3l

Ideally, our system should say:

```text
I don't have enough information in the provided documents.
```

But here's an important problem:

Our current system **always retrieves the top K documents**.

Even if they're irrelevant.

So the LLM may receive:

```text
Docker
FastAPI
Redis
```

and still try to answer.

This exposes a real RAG weakness:

> **Retrieval systems need a way to determine whether retrieved context is sufficiently relevant.**

We'll work on this much more in later retrieval days.

---

# 🟣 PART 7 — LLM AS A JUDGE

Now we can automate some of the manual evaluation.

Instead of us reading every answer:

```text
Human
 ↓
Read answer
 ↓
Score answer
```

we can use another LLM:

```text
Question
Expected answer
Generated answer
Retrieved context
        ↓
      LLM Judge
        ↓
Scores + explanation
```

---

# 🟢 CELL 22 — Create an evaluator prompt

In [54]:
def evaluate_answer_with_llm(
    question,
    context,
    expected_answer,
    generated_answer
):

    prompt = f"""
You are evaluating the quality of a RAG system.

Evaluate the generated answer using the information below.

QUESTION:
{question}

EXPECTED ANSWER:
{expected_answer}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{generated_answer}

Evaluate three dimensions.

1. Correctness:
Is the generated answer factually consistent with the expected answer?

2. Relevance:
Does the generated answer directly answer the question?

3. Faithfulness:
Is the generated answer supported by the retrieved context?

Give each score from 1 to 3.

1 = poor
2 = partially correct
3 = good

Return your evaluation in this exact format:

Correctness: <1-3>
Relevance: <1-3>
Faithfulness: <1-3>
Reason: <short explanation>
"""

    response = llm.invoke(prompt)

    return response.content

---

# 🟢 CELL 23 — Test the judge

In [55]:
result = eval_results[0]

context = "\n\n".join(
    result["sources"]
)

evaluation = evaluate_answer_with_llm(
    question=result["question"],
    context=context,
    expected_answer=result["expected_answer"],
    generated_answer=result["generated_answer"]
)

print(evaluation)

[{'type': 'text', 'text': 'Correctness: 3\nRelevance: 3\nFaithfulness: 3\nReason: The generated answer is fully accurate, directly answers the question, aligns perfectly with the expected answer, and is entirely supported by the provided context.', 'extras': {'signature': 'EqIPCp8PARFNMg/MDQ/LThByURglNYNQRRoE0Mok9tQeKqsZXy9MCGUCcB6fPD2yp5JPRYFxr9kussjTTG9LB5T1x5qD0J1deBaNEGLk9kYhncnTTBGPgWj/q87o5IvngcEOYWJXipgnF/LxT8b6ocgemf0HJ0qKoKyLoxupRSX31PFB+yADO92Qa8uvYOgFxf4I08AB6GV3gQZrxWKIjZC6oVjm+paCOME6OobdxpuHxQhTFXeL8QL4LGIpERx6qYtZEr7xY8xtbVSwa31r3DaoozOX5NsQesG5llQ71pGJMJ8eDMPSMK/VPNevZ9G4osbDp9pDkkkcX6PEqPdIZlivqHlGAREBZ+b7IMw8ty7OJJglrImi8jUKyYL8S1NvVDc9Fjm0cjLkFktcyf0XOczZZ7eV9/DKiCzqbB9C3v7Vq00bOODU74L1uoKTwsteGiG6AVfgFsshvKMvzKiqmzzgMRAuK6MzWuCphECAPnekjW/XlAtz/NmcYycu972EA4EFK7oYcOh/ooRiC+M5OMcVxHZp/3bs5JihbcEq84TJmF6tgBjwhoexBgvveN3xpvRWzJ2LOO9MS9iT5bk+gt5zdXYKJTIbsL9XnGbGn+iYwUGizZ9bXAM+3jsNgkQ95A79lrxCwwqVbYlPRCqc+2E8/EhHBeldMc2o7jIh3PHtGtZJItYOlrIZpfTy+pZlutSyY84STLzfvJGvR8wvw3

You might get something like:

```text
Correctness: 3
Relevance: 3
Faithfulness: 3
Reason: The answer directly answers the question and is supported by the retrieved context.
```

The exact wording will vary.

---

# 🧠 IMPORTANT — LLM-AS-A-JUDGE IS NOT GROUND TRUTH

This is a **very important AI engineering concept**.

Don't say:

> "The LLM gave my answer 3/3, therefore it is definitely correct."

Instead:

> "LLM-based evaluation provides scalable automated assessment, but it should be validated against human judgments and can itself make errors."

Why?

Because the judge is another model.

It can:

* misunderstand the question
* miss subtle factual errors
* prefer verbose answers
* be sensitive to wording
* produce inconsistent scores
* inherit biases from the model

So production evaluation often combines:

```text
Automated metrics
       +
LLM-based evaluation
       +
Human evaluation
```

---

# 🟢 CELL 24 — Evaluate all answers

In [56]:
llm_evaluations = []

for result in eval_results:

    context = "\n\n".join(
        result["sources"]
    )

    evaluation = evaluate_answer_with_llm(
        question=result["question"],
        context=context,
        expected_answer=result["expected_answer"],
        generated_answer=result["generated_answer"]
    )

    llm_evaluations.append({
        "question": result["question"],
        "evaluation": evaluation
    })


Print:

In [57]:
for item in llm_evaluations:

    print("=" * 70)
    print(item["question"])
    print("=" * 70)
    print(item["evaluation"])
    print()

How do Python generators save memory?
[{'type': 'text', 'text': 'Correctness: 3\nRelevance: 3\nFaithfulness: 3\nReason: The generated answer accurately and directly answers the question, fully aligns with the expected answer, and is completely supported by the retrieved context.', 'extras': {'signature': 'Ev4NCvsNARFNMg9cLuNXGWvbkQtWkRCYd+c61VoC9e3HJvV5qg5J9oCnP91OdAf5dRbZvn6ALpF9P5ApxUR/oz4tKX/GeHIVmgpoGJQWWHZq+0Oko9gQwOZ7cS/T/pnBBRikPOqu6ve+XhostCo9B3tnWtOXkB4LgfB0XY3p3EZRCZxw5hyrFW5oFRAWRqixzIDlox4WI5xZlw1XfmM03ZrOjTijElO6QfbrmK1R1UZYQdSU1Dadq/u3vIlH8g/18Nn35zkT0677hTCVthFNdmadOi1jHY25NgTBlMKmLg2XHKw3mrSfiHh48ml9TRh76nyup/JdDXpizGdREgrlQSZSVmlTFWuDnG48Y3LdOTV+0dEHAVzVKPz3A76Hd9gjGEIAFP4dz/9MTZ3Hhi4ElDCVhS6mkR8mLYsHQJU5EtHI2aDTrkvHoCzvgyXF+X+IEl5up9ojvyHTp9HvjCOZnJGGBmD47usWRJyQV6LgFK8NyW6TZFAXNZt9yt3QaTH04PnrEffieeE9EFi0yPj5HK3sIL8NIiZ4n0d2X9y61y6JCTLP+JRICYgN+6b0acEwYJqqkQFThPftJp3Bic24VzAhs/Q7SDMPH0w1BNMmIRQ5/T7z6bV5NJ0AXYWN7EI1FTACNiAeKBqetMBpJgrRkgBfBZ6TiYXhL3/cLwW+0cP3746jOMkCd

---

# 🔥 PART 8 — END-TO-END EVALUATION

Now understand the full picture.

A RAG system can fail in different places.

### Failure 1 — Retrieval failure

```text
Question
   ↓
Retriever
   ↓
❌ Wrong documents
```

Even a perfect LLM cannot answer if the correct information wasn't retrieved.

---

### Failure 2 — Context failure

```text
Retriever
   ↓
Relevant documents
   ↓
❌ Poor context construction
```

The correct information exists but isn't presented effectively.

---

### Failure 3 — Generation failure

```text
Relevant context
   ↓
LLM
   ↓
❌ Unsupported answer
```

---

### Failure 4 — Evaluation failure

```text
System looks good
        ↓
But evaluation dataset doesn't represent
real user questions
```

This is why evaluation datasets themselves matter.

---

# 🧠 Your complete Day 8 mental model

```text
                    QUESTION
                       │
                       ↓
                  RETRIEVER
                       │
             ┌─────────┴─────────┐
             ↓                   ↓
         Recall@K            Precision@K
             │
             ↓
        Retrieved Context
             │
             ↓
             LLM
             │
             ↓
        Generated Answer
             │
       ┌─────┼──────────┐
       ↓     ↓          ↓
 Correctness Relevance Faithfulness
       │     │          │
       └─────┴──────────┘
                 ↓
          END-TO-END QUALITY
```

---

# 🎯 Day 8 — What you should be able to explain

By the end of today, you should be able to answer these in an interview:

### What is Recall@K?

> Recall@K measures whether the relevant information appears within the top K retrieved results.

### What is Precision@K?

> Precision@K measures the proportion of retrieved results that are actually relevant.

### Difference?

> Recall focuses on coverage of relevant information, while precision focuses on the quality of the retrieved set.

### What is faithfulness?

> Faithfulness measures whether the generated answer is supported by the retrieved context rather than introducing unsupported information.

### What is answer relevance?

> Whether the generated response actually addresses the user's question.

### Why isn't high retrieval recall enough?

> Because retrieving the correct information doesn't guarantee that the LLM will use it correctly. Generation can still produce irrelevant or unsupported answers.

### Why use LLM-as-a-judge?

> It provides scalable automated evaluation for qualities that are difficult to measure with simple metrics, but it isn't perfect and should be validated against human judgments and can itself make errors.

---

# 📝 CELL 25 — Day 8 Reflection

Put this at the bottom of your notebook:


# Day 8 Reflection

## What I learned

- RAG systems need systematic evaluation rather than only manual testing.
- Recall@K measures whether relevant information appears in the top K retrieved results.
- Precision@K measures how much of the retrieved information is actually relevant.
- Retrieval quality and answer quality are separate problems.
- Answer correctness measures whether the generated answer is correct.
- Answer relevance measures whether the response actually answers the question.
- Faithfulness measures whether the generated answer is supported by the retrieved context.
- A RAG system can have good retrieval but still generate a poor answer.
- Evaluation datasets should contain representative questions and expected results.
- Insufficient-information handling is an important part of RAG evaluation.
- LLM-as-a-judge can automate parts of evaluation, but the judge itself can make mistakes.
- Good AI evaluation combines automated metrics, model-based evaluation, and human judgment.

## Key takeaway

Building a RAG system is only half the job.

An AI engineer also needs to measure whether the system retrieves
the right information and whether the generated answer is correct,
relevant, and grounded in that information.


---

# 🏁 DAY 8 DONE CHECKLIST

Before marking Day 8 complete, you should have:

```text
☐ Built a fresh RAG system
☐ Created an evaluation dataset
☐ Implemented Recall@K
☐ Implemented Precision@K
☐ Compared Recall vs Precision
☐ Generated answers for all evaluation questions
☐ Manually evaluated correctness
☐ Manually evaluated relevance
☐ Manually evaluated faithfulness
☐ Tested an insufficient-information question
☐ Learned LLM-as-a-Judge
☐ Ran an LLM evaluator
☐ Understood why LLM judges aren't perfect
☐ Added Day 8 reflection
☐ Pushed notebook to GitHub
```

**This is the complete Day 8 notebook.** And importantly, it is **standalone** — you can create a brand-new Colab, paste the cells from top to bottom, and it won't depend on anything from your deleted notebook.